In [1]:
from pydantic_ai import format_as_xml
from setup_quarto import setup_dot_env

ctx = setup_dot_env()

/Users/mattijnvanhoek/hkv/talk-with-fews/.pixi/envs/default/lib/python3.13/site-packages/pydantic/fields.py:1093: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warn(


### XML Validation

In [2]:
# xml snippet on root level with direct parent with missing nested elements
xml_snippet = """
<series>
    <header>
        <type>instantaneous</type>
        
        <parameterId>WaterLevel</parameterId>
        <timeStep unit="second" multiplier="3600"/>
        <startDate date="2006-08-23" time="17:00:00"/>
        <endDate date="2006-08-24" time="00:00:00"/>
        <missVal>-8888.0</missVal>
    </header>
    <event date="2006-08-23" time="17:00:00" value="8.66"/>
</series>
"""
schema_url = "data/fews_schemas/fews_wldelft_nl/pi-schemas/pi_timeseries.xsd"

In [3]:
from talk_with_fews.tools.validator.validate_xml import validate_xml
from talk_with_fews.models.validate_xml import XMLValidationOutput, XMLValidationInput

xml_validation_input = XMLValidationInput(
    xml_content=xml_snippet, schema_url=schema_url
)
xml_validation_output: XMLValidationOutput = await validate_xml(
    ctx=ctx, input=xml_validation_input
)

In [4]:
print(format_as_xml(xml_validation_output))

<is_valid>False</is_valid>
<errors>
  <item>
    <reason>Unexpected child with tag 'parameterId' at position 2. Tag 'locationId' expected.</reason>
    <xpath>Found direct parent 'TimeSeries' for 'series'</xpath>
  </item>
</errors>
<warnings />
<schema_location>data/fews_schemas/fews_wldelft_nl/pi-schemas/pi_timeseries.xsd</schema_location>


In [5]:
# xml snippet with indirect parent to root with missing nested elements
xml_snippet = """
	<general>
		<exportType>SomeValidExportType</exportType>
		<folder>MyExportFolder</folder>
		<exportFileName>
			<prefix>
				<timeZeroFormattingString>yyyyMMddHHmm</timeZeroFormattingString>
			</prefix>
		</exportFileName>
		<idMapId>MyIdMap</idMapId>
		<forecastSelectionPeriod end="0" unit="day"/>
	</general>
"""
schema_url = "data/fews_schemas/fews_wldelft_nl/timeSeriesExportRun.xsd"

xml_validation_input = XMLValidationInput(
    xml_content=xml_snippet, schema_url=schema_url
)
xml_validation_output: XMLValidationOutput = await validate_xml(
    ctx=ctx, input=xml_validation_input
)

print(format_as_xml(xml_validation_output))

<is_valid>False</is_valid>
<errors>
  <item>
    <reason>Validation failed: {'end': '0', 'unit': 'day'} with XsdAttributeGroup(['start', 'end', 'startOverrulable', 'endOverrulable', 'unit']):

Reason: missing required attribute 'start'</reason>
    <xpath>Found indirect parent 'timeSeriesExportRun' for 'general'. Child 'export' can contain 'general'</xpath>
  </item>
</errors>
<warnings />
<schema_location>data/fews_schemas/fews_wldelft_nl/timeSeriesExportRun.xsd</schema_location>


### XML Templating

In [6]:
# partial xml template generation

from talk_with_fews.tools.validator.generate_xml_template import generate_xml_template
from talk_with_fews.models.generate_xml_template import (
    XMLTemplateInput,
    XMLTemplateOutput,
)

xml_template_input = XMLTemplateInput(schema_url=schema_url)
xml_template_output: XMLTemplateOutput = await generate_xml_template(
    ctx=ctx, input=xml_template_input
)

print(xml_template_output.template)

<timeSeriesExportRun xmlns="http://www.wldelft.nl/fews">
<timeSeriesExportRun>
  <export>
    <general>
      <exportTypeStandard>example_value</exportTypeStandard>
      <exportType>example_value</exportType>
      <serializerClassName>example_value</serializerClassName>
      <folder>example_value</folder>
      <exportFileName>
        <name>example_value</name>
        <useExternalLocationIdAsName>true</useExternalLocationIdAsName>
      </exportFileName>
      <serverUrl>example_value</serverUrl>
      <user>example_value</user>
      <password>example_value</password>
      <encryptedPassword>example_value</encryptedPassword>
      <exportMissingValue>1.0</exportMissingValue>
      <exportMissingValueString>example_value</exportMissingValueString>
      <dataFeedId>example_value</dataFeedId>
      <disableDataFeedInfo/>
      <columnSeparator>example_value</columnSeparator>
      <decimalSeparator>1.0</decimalSeparator>
    </general>
    <annotationLocationSetId>example_value</a

In [7]:
# full xml template generation
from talk_with_fews.tools.validator.generate_xml_template import generate_xml_template
from talk_with_fews.models.generate_xml_template import (
    XMLTemplateInput,
    XMLTemplateOutput,
)

xml_template_input = XMLTemplateInput(schema_url=schema_url, include_optional=True)
xml_template_output: XMLTemplateOutput = await generate_xml_template(
    ctx=ctx, input=xml_template_input
)

In [8]:
print(xml_template_output.template)

<timeSeriesExportRun xmlns="http://www.wldelft.nl/fews">
<timeSeriesExportRun>
  <export>
    <general>
      <description>example_value</description>
      <exportTypeStandard>example_value</exportTypeStandard>
      <exportType>example_value</exportType>
      <serializerClassName>example_value</serializerClassName>
      <binDir>example_value</binDir>
      <moduleDataSetName>example_value</moduleDataSetName>
      <folder>example_value</folder>
      <exportFileName>
        <name>example_value</name>
        <useExternalLocationIdAsName>true</useExternalLocationIdAsName>
        <prefix>
          <simpleString>example_value</simpleString>
          <timeZeroFormattingString>2024-01-01</timeZeroFormattingString>
          <currentTimeFormattingString>2024-01-01</currentTimeFormattingString>
        </prefix>
        <suffix>
          <simpleString>example_value</simpleString>
          <timeZeroFormattingString>2024-01-01</timeZeroFormattingString>
          <currentTimeFormattin